# 1. Import Libraries


In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_class_weight

from catboost import CatBoostClassifier
import optuna

# 2. Load Dataset

In [2]:
TRAIN_PATH = "../data/train.csv"
TEST_PATH = "../data/test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print(f"Train Shape : {train.shape}")
print(f"Test Shape  : {test.shape}")


Train Shape : (690088, 15)
Test Shape  : (295753, 14)


In [3]:
# ============================
# Prepare Data
# ============================

TARGET = "health_condition"
ID_COL = "id"

X = train.drop(columns=[TARGET])
y = train[TARGET]

X_test = test.copy()

# Remove ID column
if ID_COL in X.columns:
    X = X.drop(columns=[ID_COL])

if ID_COL in X_test.columns:
    X_test = X_test.drop(columns=[ID_COL])

print("Training Features :", X.shape)
print("Training Labels   :", y.shape)
print("Test Features     :", X_test.shape)

Training Features : (690088, 13)
Training Labels   : (690088,)
Test Features     : (295753, 13)


In [4]:
# ============================
# Identify Feature Types
# ============================

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_features = X.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

cat_feature_indices = [
    X.columns.get_loc(col)
    for col in categorical_features
]

print(f"Categorical Features : {len(categorical_features)}")
print(f"Numerical Features   : {len(numerical_features)}")

print(categorical_features)

Categorical Features : 6
Numerical Features   : 7
['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']


In [5]:
# ============================
# Optuna Objective Function
# ============================

def objective(trial):

    params = {
        "iterations": trial.suggest_int("iterations", 1000, 4000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "depth": trial.suggest_int("depth", 5, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 20),
        "random_strength": trial.suggest_float("random_strength", 0, 10),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0, 5),
        "border_count": trial.suggest_int("border_count", 64, 255),

        "loss_function": "MultiClass",
        "eval_metric": "TotalF1",

        "task_type": "GPU",
        "devices": "0",

        "random_seed": 42,
        "verbose": False
    }

    skf = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    scores = []

    for train_idx, valid_idx in skf.split(X, y):

        X_train = X.iloc[train_idx].copy()
        X_valid = X.iloc[valid_idx].copy()

        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        # Fill missing values
        for col in categorical_features:
            X_train[col] = X_train[col].fillna("Missing")
            X_valid[col] = X_valid[col].fillna("Missing")

        for col in numerical_features:
            median = X_train[col].median()
            X_train[col] = X_train[col].fillna(median)
            X_valid[col] = X_valid[col].fillna(median)

        # Class weights
        classes = np.unique(y_train)

        weights = compute_class_weight(
            class_weight="balanced",
            classes=classes,
            y=y_train
        )

        class_weights = dict(zip(classes, weights))

        model = CatBoostClassifier(
            **params,
            class_weights=class_weights
        )

        model.fit(
            X_train,
            y_train,
            eval_set=(X_valid, y_valid),
            cat_features=cat_feature_indices,
            early_stopping_rounds=300,
            use_best_model=True
        )

        preds = model.predict(X_valid)

        score = balanced_accuracy_score(y_valid, preds)

        scores.append(score)

    return np.mean(scores)

In [9]:
# ============================
# Run Optuna Study
# ============================

study = optuna.create_study(
    direction="maximize",
    study_name="CatBoost_Optimization"
)

study.optimize(
    objective,
    n_trials=50,          # Increase to 100+ later
    show_progress_bar=True
)

[I 2026-07-26 19:20:13,751] A new study created in memory with name: CatBoost_Optimization


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-07-26 19:21:33,083] Trial 0 finished with value: 0.9489801297899813 and parameters: {'iterations': 1510, 'learning_rate': 0.03962472501753017, 'depth': 5, 'l2_leaf_reg': 4.561800552269297, 'random_strength': 9.223917172278638, 'bagging_temperature': 2.0323764880325146, 'border_count': 183}. Best is trial 0 with value: 0.9489801297899813.
[I 2026-07-26 19:23:35,793] Trial 1 finished with value: 0.9482965192524698 and parameters: {'iterations': 2085, 'learning_rate': 0.013507583684791347, 'depth': 5, 'l2_leaf_reg': 9.437664852017345, 'random_strength': 6.990095537743127, 'bagging_temperature': 1.80758405407276, 'border_count': 67}. Best is trial 0 with value: 0.9489801297899813.
[I 2026-07-26 19:27:08,117] Trial 2 finished with value: 0.94815586632674 and parameters: {'iterations': 2711, 'learning_rate': 0.0202679324172851, 'depth': 9, 'l2_leaf_reg': 3.912883929841384, 'random_strength': 0.8245224648858529, 'bagging_temperature': 4.170574696783086, 'border_count': 99}. Best is tr

In [12]:
# ============================
# Best Trial
# ============================

print("=" * 60)
print("Best Balanced Accuracy :", study.best_value)
print("=" * 60)

print("\nBest Parameters:\n")

for key, value in study.best_params.items():
    print(f"{key}: {value}")

Best Balanced Accuracy : 0.9493669609907288

Best Parameters:

iterations: 3501
learning_rate: 0.07298197601178044
depth: 8
l2_leaf_reg: 2.6130877247254016
random_strength: 1.9111220326055867
bagging_temperature: 0.5503784078725401
border_count: 244


In [14]:
# ============================
# Save Best Parameters
# ============================

best_params = study.best_params

pd.DataFrame([best_params]).to_csv(
    "best_catboost_params.csv",
    index=False
)

print("Best parameters saved successfully!")

Best parameters saved successfully!


In [15]:
import json
import os

os.makedirs("../output", exist_ok=True)

with open("../output/best_params.json", "w") as f:
    json.dump(study.best_params, f, indent=4)

print("Best parameters saved successfully!")

Best parameters saved successfully!


In [6]:
from catboost import CatBoostClassifier

# Final model
final_model = CatBoostClassifier(
    iterations=3501,
    learning_rate=0.07298197601178044,
    depth=8,
    l2_leaf_reg=2.6130877247254016,
    random_strength=1.9111220326055867,
    bagging_temperature=0.5503784078725401,
    border_count=244,

    loss_function="MultiClass",
    eval_metric="TotalF1",
    task_type="GPU",
    devices="0",

    random_seed=42,
    verbose=200
)

# Train on the full dataset
final_model.fit(
    X,
    y,
    cat_features=cat_feature_indices
)

CatBoostError: Invalid type for cat_feature[non-default value idx=254,feature_idx=7]=nan : cat_features must be integer or string, real number values and NaN values should be converted to string.